In [1]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors

In [2]:
df_main = pd.read_csv("~/mission_sih/mission_sih/data/nasa_data_cleaned.csv")
df_main = df_main[df_main["type"].isin([3])]
df_main["type"].value_counts()

type
3    10508
Name: count, dtype: int64

In [ ]:
df_cat = pd.read_csv("~/mission_sih/mission_sih/data/land_use_data.csv")

In [3]:
df_cat = pd.read_csv("~/mission_sih/mission_sih/data/osm_data_cleaned.csv")

In [ ]:
df_main['acq_date'] = pd.to_datetime(df_main['acq_date'])
df_main['year'] = df_main['acq_date'].dt.year

In [36]:

EARTH_RADIUS_M = 6371000
RADIUS_LIMIT_M = 2000

In [37]:
df_main['category'] = pd.Series([np.nan] * len(df_main), dtype='object')
df_main['match_dist_m'] = np.nan

In [ ]:
for yr, group in df_main.groupby('year'):
    cat_group = df_cat[df_cat['year'] == yr]   
    if cat_group.empty:
        continue  # no category data for this year, leave unmatched

    cat_rad = np.radians(cat_group[['latitude', 'longitude']].values)
    main_rad = np.radians(group[['latitude', 'longitude']].values)

    nn = NearestNeighbors(n_neighbors=1, metric='haversine', n_jobs=-1)
    nn.fit(cat_rad)
    dist, idx = nn.kneighbors(main_rad)

    dist_m = dist.flatten() * EARTH_RADIUS_M
    matched_cat = cat_group.iloc[idx.flatten()]['land_type'].values.astype(object)

    # apply 2km cutoff
    matched_cat = np.where(dist_m <= RADIUS_LIMIT_M, matched_cat, np.nan)

    df_main.loc[group.index, 'category'] = matched_cat
    df_main.loc[group.index, 'match_dist_m'] = dist_m

In [38]:
# df _cat2 has lat, lon, category — no year column, direct global search
cat_rad = np.radians(df_cat[['latitude', 'longitude']].values)
main_rad = np.radians(df_main[['latitude', 'longitude']].values)
nn = NearestNeighbors(n_neighbors=1, metric='haversine', n_jobs=-1)
nn.fit(cat_rad)
dist, idx = nn.kneighbors(main_rad)
dist_m = dist.flatten()*EARTH_RADIUS_M
matched_cat = df_cat.iloc[idx.flatten()]['osm_category'].values.astype(str)

valid = dist_m <= RADIUS_LIMIT_M
# overwrite only if: valid AND (no previous match OR this one is closer)
no_prev_match = df_main['category'].isna().values
df_main.loc[valid, 'category'] = matched_cat[valid]
df_main.loc[valid, 'match_dist_m'] = dist_m[valid]

In [39]:
df_main["category"].value_counts(dropna=False)

category
industrial_facility    8572
NaN                    1764
quarry                   71
gas_flare                64
power_plant              37
Name: count, dtype: int64

In [40]:
df_main = df_main.dropna(subset=["category"])
df_main.reset_index(inplace=True)

In [41]:

df_main.drop(columns=["index"],inplace=True)

In [42]:
df = df_main.copy()

In [44]:
popped_column = df.pop("match_dist_m")

df.insert(12, "match_dist_m", popped_column)


In [45]:
df.head()

,latitude,longitude,brightness,scan,track,acq_date,acq_time,confidence,bright_t31,frp,daynight,type,match_dist_m,category
0,22.79488,86.20175,303.88,0.39,0.36,2018-04-02,1956,1,291.97,1.12,1,3,823.705235,industrial_facility
1,22.79219,86.19742,309.66,0.39,0.36,2018-04-02,1956,1,294.21,1.04,1,3,821.842283,industrial_facility
2,21.18458,81.38937,312.06,0.49,0.41,2018-04-02,1956,1,297.98,1.56,1,3,106.958488,industrial_facility
3,11.63223,92.75088,336.30,0.38,0.36,2018-04-03,705,1,303.66,3.53,0,3,305.551027,quarry
4,23.55042,87.24222,308.04,0.46,0.39,2018-04-03,1937,1,294.05,1.31,1,3,661.567717,industrial_facility


In [46]:
df.to_csv("~/mission_sih/mission_sih/data/type_3_dataset.csv", index=False)

In [ ]:
df.info()